In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import sawtooth,square
import time

In [2]:
DAC_SR = 9.8304e9 #in GHz
DAC_amplitude = 2**15 #16 bit representation in 2's complement format
N_points = 2**18
# duration = 10e-6  # total waveform duration in seconds (example: 20 µs)
# N_points = int(DAC_SR * duration) // 8 * 8
MAX_POINTS = int(2**14)
t = 1/DAC_SR*np.arange(N_points)
print(N_points)
print(MAX_POINTS)
print(t)

262144
16384
[0.00000000e+00 1.01725260e-10 2.03450521e-10 ... 2.66663615e-05
 2.66664632e-05 2.66665649e-05]


In [3]:
duration = 6.65e-6
N_points = int(DAC_SR * duration) // 8 * 8
MAX_POINTS = int(N_points / 16)
t = 1/DAC_SR*np.arange(N_points)
print(N_points)
print(MAX_POINTS)
print(t)

65368
4085
[0.00000000e+00 1.01725260e-10 2.03450521e-10 ... 6.64927165e-06
 6.64937337e-06 6.64947510e-06]


In [4]:
def dBm2Vpp(dbm):
    return 0.63245*10**(dbm/20)

print(dBm2Vpp(-6))
print(2*np.sqrt(100)*10**(-3/2))

0.3169758659075683
0.6324555320336758


In [5]:
#Generating test signal
freq = 1.02e9 #in Hz
num_of_zeros = 2**18-N_points
####### waveforms here
#DAC_B
mysignal_0z = 2*dBm2Vpp(-25)*DAC_amplitude*np.sin(2 * np.pi * 0.02e9 * t)
mysignal_0z = np.concatenate((mysignal_0z,np.zeros(num_of_zeros)))

mysignal_0x =  2*dBm2Vpp(-15)*DAC_amplitude*np.sin(2 * np.pi * 1 * 0.04e9 * t)
mysignal_0x = np.concatenate((mysignal_0x,np.zeros(num_of_zeros)))

#DAC_A
mysignal_2z = 2*dBm2Vpp(-20)*DAC_amplitude*np.sin(2 * np.pi * 1 * 0.06e9 * t)
mysignal_2z = np.concatenate((mysignal_2z,np.zeros(num_of_zeros)))

mysignal_2x =  2*dBm2Vpp(-10)*DAC_amplitude*np.sin(2 * np.pi * 1 * 0.07e9 * t)
mysignal_2x = np.concatenate((mysignal_2x,np.zeros(num_of_zeros)))

####################


total_waveform_0 = np.append(mysignal_0z,mysignal_0x)

mask_0 = ([True]*8+[False]*8)*(len(total_waveform_0)//16)
mask_1 = ([False]*8+[True]*8)*(len(total_waveform_0)//16)

RAM_0 = total_waveform_0[mask_0]
RAM_1 = total_waveform_0[mask_1]
mysignal_0 = np.append(RAM_0,RAM_1)

total_waveform_2 = np.append(mysignal_2z,mysignal_2x)

mask_2 = ([True]*8+[False]*8)*(len(total_waveform_2)//16)
mask_3 = ([False]*8+[True]*8)*(len(total_waveform_2)//16)

RAM_2 = total_waveform_2[mask_2]
RAM_3 = total_waveform_2[mask_3]
mysignal_1 = np.append(RAM_2,RAM_3)

mysignal = np.append(mysignal_0,mysignal_1)
mysignal = mysignal.astype(np.int16)


In [8]:
len(mysignal)/2**20

1.0

In [74]:
RAM_1

array([  237.94552109,   267.56470379,   297.14016447, ...,
       -1341.71301285, -1156.00242367,  -969.53626845])

### Load the overlay into the FPGA and program the oscillators on the board to produce the appropriate clock signals. 
The LMK and LMX files were taken from https://github.com/Xilinx/RFSoC-PYNQ/tree/master/boards/RFSoC4x2/packages/tics/tics/register_txts

In [97]:
from pynq import Overlay
import xrfclk
from pynq import PL
import pynq
from pynq.lib import AxiGPIO
PL.reset()
ol = Overlay("./arb_dual_DMA_215MHz_noClockSpikes.bit") #Load the FPGA bit file 
xrfclk.set_ref_clks(lmk_freq = 245.76,lmx_freq = 491.52) #Programs the oscillators on the board to produce the appropriate clock signals

In [113]:
rst_n_ip = ol.ip_dict['rst_n']
rst_n =  AxiGPIO(rst_n_ip).channel1
print("Asserting extenal reset (low)...")
rst_n.write(0,1)
time.sleep(0.1)
print('Releasing external reset (high)...')
rst_n.write(1,1)

write_enable_ip = ol.ip_dict['write_enable']
write_enable = AxiGPIO(write_enable_ip).channel1
write_enable.write(1,0x1)

dma = ol.axi_dma_0
ndata = len(mysignal)
buf = pynq.allocate(shape = (ndata),dtype = np.uint16)
buf[:] = mysignal[:]
start = time.time()
dma.sendchannel.transfer(buf)
dma.sendchannel.wait()
stop = time.time()
print(f"Time spent wiriting : {stop - start}s")

max_instance = ol.ip_dict['MAX_POINTS']
max_port = AxiGPIO(max_instance).channel1
max_port.write(int(MAX_POINTS),0xffffffff)

axi_control_instance = ol.ip_dict['axi_control']
axi_control = AxiGPIO(axi_control_instance).channel1
axi_control.write(0,0x1)

enable_ch0_instance = ol.ip_dict['enable_ch0']
enable_ch0 = AxiGPIO(enable_ch0_instance).channel1

enable_ch2_instance = ol.ip_dict['enable_ch2']
enable_ch2 = AxiGPIO(enable_ch2_instance).channel1

write_enable.write(0,0x1)

Asserting extenal reset (low)...
Releasing external reset (high)...
Time spent wiriting : 0.0070688724517822266s


In [ ]:
axi_control.write(1,0x1)
enable_ch2.write(1,0x1)
enable_ch0.write(1,0x1)

In [15]:
write_enable.write(0,0x1)

In [17]:
from pynq import ps

print("CPU Clock Frequency:", ps.Clocks.cpu_mhz, "MHz")
print("FCLK0 Clock Frequency:", ps.Clocks.fclk0_mhz, "MHz")
print("FCLK1 Clock Frequency:", ps.Clocks.fclk1_mhz, "MHz")
print("FCLK2 Clock Frequency:", ps.Clocks.fclk2_mhz, "MHz")
print("FCLK3 Clock Frequency:", ps.Clocks.fclk3_mhz, "MHz")

CPU Clock Frequency: 1199.988 MHz
FCLK0 Clock Frequency: 214.283571 MHz
FCLK1 Clock Frequency: 99.999 MHz
FCLK2 Clock Frequency: 99.999 MHz
FCLK3 Clock Frequency: 99.999 MHz
